In [9]:
# Cell 1 — Install Dependencies
# pin datasets to last version supporting legacy loading scripts (wiki_dpr needs it)
!pip install -q \
    "datasets==2.14.7" \
    transformers \
    faiss-cpu \
    tqdm


In [10]:
# Cell 2 — Imports + Setup
import json
import re
import string
import os
from pathlib import Path

import numpy as np
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [11]:
# Cell 3 — Normalize (matches utils.py normalize_str exactly)
def normalize_str(text: str) -> str:
    tokens = text.lower().split()
    articles = {'a', 'an', 'the'}
    cleaned = []
    for token in tokens:
        t = token.strip(string.punctuation)
        if t and t not in articles:
            cleaned.append(t)
    return ' '.join(cleaned)

In [12]:
# Cell 4 — Load NQ Queries + Pre-normalize Answers
def _load_nq_answers(n=50):
    # load NQ once, return queries and normalized answer lists together
    nq = load_dataset("nq_open", split="validation")
    queries = []
    nq_answers = []
    for i in range(n):
        ex = nq[i]
        answers = ex["answer"] if isinstance(ex["answer"], list) else [ex["answer"]]
        queries.append({
            "id": f"nq_dev_{i}",
            "question": ex["question"],
            "answers": answers,
            "gold_passage_id": None,
        })
        nq_answers.append([
            normalize_str(a) for a in answers
            if len(normalize_str(a)) >= 3
        ])
    return queries, nq_answers

In [13]:
# Cell 5 — Build Passage Corpus (original wiki_dpr — works with datasets==2.14.7)
def build_passage_corpus(target_size=30000):
    print("Loading NQ queries...")
    queries, nq_answers = _load_nq_answers(50)

    all_answers_flat = {
        ans for ans_list in nq_answers for ans in ans_list
    }

    print("Streaming wiki_dpr (psgs_w100.nq.no_index)...")
    dataset = load_dataset(
    "wiki_dpr",
    "psgs_w100.nq.no_index",
    split="train",
    streaming=True
    )


    gold_passages = {}
    other_passages = []

    pbar = tqdm(desc="Collecting passages", unit="passages")
    for example in dataset:
        pid = str(example["id"])
        text = example["text"]
        norm_text = normalize_str(text)

        is_gold = (
            pid not in gold_passages
            and any(ans in norm_text for ans in all_answers_flat)
        )

        if is_gold:
            gold_passages[pid] = {"id": pid, "text": text}
        elif len(other_passages) < target_size:
            other_passages.append({"id": pid, "text": text})

        pbar.update(1)

        if len(gold_passages) + len(other_passages) >= target_size:
            break

    pbar.close()

    passages = list(gold_passages.values())
    remaining = target_size - len(passages)
    passages.extend(other_passages[:remaining])

    print(f"Corpus: {len(passages)} passages "
          f"({len(gold_passages)} gold, "
          f"{len(passages) - len(gold_passages)} random)")

    print("Matching gold passage IDs...")
    norm_texts = {p["id"]: normalize_str(p["text"]) for p in passages}
    for q, ans_list in zip(queries, nq_answers):
        if not ans_list:
            continue
        for p in passages:
            if any(ans in norm_texts[p["id"]] for ans in ans_list):
                q["gold_passage_id"] = p["id"]
                break

    found = sum(1 for q in queries if q["gold_passage_id"] is not None)
    print(f"Gold passages found: {found}/50")

    return passages, queries


In [14]:
# Cell 6 — Contriever Embeddings (GPU + masked pooling + AMP)
def compute_contriever_embeddings(passages, batch_size=None, checkpoint_every=5000):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if batch_size is None:
        batch_size = 64 if device == "cuda" else 16
    print(f"Device: {device}, batch_size: {batch_size}")

    texts = [p["text"] if isinstance(p, dict) else p for p in passages]
    n = len(texts)

    # resume from latest checkpoint if available
    start_idx = 0
    existing_embeddings = []
    checkpoints = sorted(DATA_DIR.glob("embeddings_checkpoint_*.npy"))
    if checkpoints:
        latest = checkpoints[-1]
        try:
            ckpt = np.load(latest)
            start_idx = ckpt.shape[0]
            existing_embeddings = [ckpt]
            print(f"Resuming from checkpoint: {latest} ({start_idx}/{n})")
        except Exception as e:
            print(f"Could not load checkpoint: {e}, starting fresh")

    if start_idx >= n:
        print("Embeddings already complete")
        return np.concatenate(existing_embeddings, axis=0)

    print("Loading Contriever...")
    tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")
    model = AutoModel.from_pretrained("facebook/contriever").to(device)
    model.eval()

    embedding_chunks = []
    processed = 0
    pbar = tqdm(total=n - start_idx, desc="Computing embeddings")

    for i in range(start_idx, n, batch_size):
        batch = texts[i: i + batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True,
            max_length=128, return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            if device == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = model(**inputs)
            else:
                outputs = model(**inputs)

        # masked mean pooling — excludes padding tokens
        token_emb = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).expand(token_emb.size()).float()
        batch_emb = (token_emb * mask).sum(dim=1) / mask.sum(dim=1)
        embedding_chunks.append(batch_emb.cpu().numpy())

        processed += len(batch)
        pbar.update(len(batch))

        if processed % checkpoint_every < batch_size:
            partial = np.concatenate(existing_embeddings + embedding_chunks, axis=0)
            ckpt_path = DATA_DIR / f"embeddings_checkpoint_{start_idx + processed}.npy"
            np.save(ckpt_path, partial)
            print(f"\nCheckpoint saved: {ckpt_path}")

    pbar.close()

    all_embeddings = np.concatenate(existing_embeddings + embedding_chunks, axis=0)

    # L2-normalize for FAISS IndexFlatIP
    norms = np.linalg.norm(all_embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return (all_embeddings / norms).astype(np.float32)

In [15]:
# Cell 7 — Save + Verify Outputs
def save_passages_and_embeddings(passages, embeddings):
    passages_path = DATA_DIR / "passages.json"
    with open(passages_path, "w") as f:
        json.dump(passages, f)

    embeddings_path = DATA_DIR / "embeddings.npy"
    np.save(embeddings_path, embeddings)

    print(f"Saved passages  -> {passages_path}")
    print(f"Saved embeddings -> {embeddings_path}")

    assert embeddings.shape == (len(passages), 768), f"Wrong shape: {embeddings.shape}"
    assert embeddings.dtype == np.float32, f"Wrong dtype: {embeddings.dtype}"
    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-5), "Not L2-normalized"
    print("Verification passed")

In [16]:
# Cell 8 — Run Pipeline
passages, queries = build_passage_corpus(target_size=30000)

queries_path = DATA_DIR / "queries.json"
with open(queries_path, "w") as f:
    json.dump(queries, f)
print(f"Saved queries    -> {queries_path}")

embeddings = compute_contriever_embeddings(passages)
save_passages_and_embeddings(passages, embeddings)

print("\nDone!")

Loading NQ queries...
Streaming wiki_dpr (psgs_w100.nq.no_index)...


Corpus: 30000 passages (20150 gold, 9850 random)
Matching gold passage IDs...
Gold passages found: 26/50
Saved queries    -> /kaggle/working/data/queries.json
Device: cuda, batch_size: 64
Loading Contriever...


tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...


Checkpoint saved: /kaggle/working/data/embeddings_checkpoint_20032.npy



Computing embeddings:  84%|████████▎ | 25088/30000 [00:44<00:09, 502.08it/s]


Checkpoint saved: /kaggle/working/data/embeddings_checkpoint_25024.npy



Computing embeddings: 100%|██████████| 30000/30000 [00:52<00:00, 566.32it/s]



Checkpoint saved: /kaggle/working/data/embeddings_checkpoint_30000.npy
Saved passages  -> /kaggle/working/data/passages.json
Saved embeddings -> /kaggle/working/data/embeddings.npy
Verification passed

Done!


In [17]:
# Cell 9 — Download outputs
# Option A: copy to Kaggle output folder (appears in Output tab on the right sidebar)
import shutil

os.makedirs("/kaggle/working/output", exist_ok=True)
for fname in ["passages.json", "embeddings.npy", "queries.json"]:
    shutil.copy(DATA_DIR / fname, f"/kaggle/working/output/{fname}")
    size = os.path.getsize(f"/kaggle/working/output/{fname}") / 1e6
    print(f"{fname}  —  {size:.1f} MB")

print("\nFiles ready — download from the Output tab in the right sidebar")

passages.json  —  19.8 MB
embeddings.npy  —  92.2 MB
queries.json  —  0.0 MB

Files ready — download from the Output tab in the right sidebar
